In [1]:
import time
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from datetime import datetime, timezone
from typing import List, Tuple
import json
import re
import pytz
import hashlib
from urllib.parse import urlparse

# --------------------
# Configuration
# --------------------

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CRAWL_DELAY = 1.5  # seconds
MAX_URLS = 200

NS = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}

# --------------------
# Sitemap crawling
# --------------------
def normalize_text(text: str) -> str:
    if not text:
        return text

    replacements = {
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
        "‚": "'",
        "‛": "'",
        "–": "-",
        "—": "-",
        "―": "-",
        "…": "...",
        "\u00a0": " ",
        "\u200b": "",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return text
    
def fetch_xml(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text


def parse_lastmod(value: str) -> datetime:
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except Exception:
        return datetime.min


def crawl_sitemap(sitemap_url, seen=None) -> List[Tuple[str, datetime]]:
    if seen is None:
        seen = set()

    if sitemap_url in seen:
        return []

    seen.add(sitemap_url)
    print(f"Fetching sitemap: {sitemap_url}")

    xml_text = fetch_xml(sitemap_url)
    root = ET.fromstring(xml_text)

    results = []

    sitemap_tags = root.findall("ns:sitemap", NS)
    if sitemap_tags:
        for sitemap in sitemap_tags:
            loc = sitemap.find("ns:loc", NS)
            if loc is not None:
                time.sleep(CRAWL_DELAY)
                results.extend(crawl_sitemap(loc.text.strip(), seen))
        return results

    for url in root.findall("ns:url", NS):
        loc = url.find("ns:loc", NS)
        lastmod = url.find("ns:lastmod", NS)

        if loc is None:
            continue

        lastmod_dt = (
            parse_lastmod(lastmod.text.strip())
            if lastmod is not None and lastmod.text
            else datetime.min
        )

        results.append((loc.text.strip(), lastmod_dt))

    return results


# --------------------
# Article extraction
# --------------------

def fetch_html(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text


def get_slug_hash(url: str) -> str:
    """
    Extract slug from URL and return its hash.
    Example: https://www.thedailybeast.com/trump-posts-vile-video -> hash of 'trump-posts-vile-video'
    """
    parsed = urlparse(url)
    path = parsed.path.strip('/')
    
    # Get the last part of the path (the slug)
    slug = path.split('/')[-1] if '/' in path else path
    
    # Return SHA256 hash of the slug
    return hashlib.sha256(slug.encode('utf-8')).hexdigest()


def normalize_published(date_str: str) -> str | None:
    """
    Extract and normalize date from formats like:
    - Feb. 9 2026 4:34PM EST
    - Feb. 9 20265:37PM EST (with seconds)
    Returns ISO-8601 UTC format
    """
    try:
        # Clean up the string - remove extra whitespace
        cleaned = date_str.strip()
        
        # Remove timezone abbreviation to parse separately
        timezone_match = re.search(r'\s+(EST|EDT|PST|PDT|CST|CDT|MST|MDT)\s*$', cleaned)
        tz_abbr = timezone_match.group(1) if timezone_match else None
        
        # Remove timezone from string
        cleaned = re.sub(r'\s+(?:EST|EDT|PST|PDT|CST|CDT|MST|MDT)\s*$', '', cleaned).strip()
        
        # Try parsing with seconds first, then without
        dt = None
        formats_to_try = [
            "%b. %d %Y%I:%M:%S%p",  # Feb. 9 20265:37:00PM (with seconds, no space before time)
            "%b. %d %Y %I:%M:%S%p",  # Feb. 9 2026 5:37:00PM (with seconds)
            "%b. %d %Y%I:%M%p",      # Feb. 9 20265:37PM (no space before time)
            "%b. %d %Y %I:%M%p",     # Feb. 9 2026 4:34PM (standard format)
        ]
        
        for fmt in formats_to_try:
            try:
                dt = datetime.strptime(cleaned, fmt)
                break
            except ValueError:
                continue
        
        if dt is None:
            raise ValueError(f"Could not parse date: {cleaned}")
        
        # Convert to UTC based on timezone
        if tz_abbr in ('EST', 'EDT'):
            eastern = pytz.timezone("US/Eastern")
            dt = eastern.localize(dt)
            return dt.astimezone(pytz.UTC).isoformat()
        elif tz_abbr in ('PST', 'PDT'):
            pacific = pytz.timezone("US/Pacific")
            dt = pacific.localize(dt)
            return dt.astimezone(pytz.UTC).isoformat()
        elif tz_abbr in ('CST', 'CDT'):
            central = pytz.timezone("US/Central")
            dt = central.localize(dt)
            return dt.astimezone(pytz.UTC).isoformat()
        elif tz_abbr in ('MST', 'MDT'):
            mountain = pytz.timezone("US/Mountain")
            dt = mountain.localize(dt)
            return dt.astimezone(pytz.UTC).isoformat()
        else:
            # Default to Eastern if no timezone
            eastern = pytz.timezone("US/Eastern")
            dt = eastern.localize(dt)
            return dt.astimezone(pytz.UTC).isoformat()
            
    except Exception as e:
        print(f"Date parsing error for '{date_str}': {e}")
        return None


def extract_body_from_html(soup) -> str:
    """Extract article body from Daily Beast HTML structure"""
    body_text = ""
    
    # Daily Beast uses div.b-article-body with article tag inside
    article_body = soup.find("div", class_="b-article-body")
    if article_body:
        article = article_body.find("article")
        if article:
            # Extract all paragraphs with class c-paragraph
            paragraphs = article.find_all("p", class_="c-paragraph")
            if paragraphs:
                body_text = "\n\n".join(get_text_with_spacing(p) for p in paragraphs if get_text_with_spacing(p))
    
    # Fallback: try to find article tag directly
    if not body_text:
        article = soup.find("article", id="main-article")
        if article:
            paragraphs = article.find_all("p", class_="c-paragraph")
            body_text = "\n\n".join(get_text_with_spacing(p) for p in paragraphs if get_text_with_spacing(p))
    
    return body_text


def get_text_with_spacing(element) -> str:
    """
    Extract text from an element with proper spacing around inline tags.
    Ensures tags like <a>, <i>, <em>, <strong> have spaces around them.
    """
    if element is None:
        return ""
    
    # Get all text with separator to ensure spacing
    text = element.get_text(separator=" ", strip=True)
    
    # Clean up multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


def is_registration_walled(soup) -> bool:
    """Check if article has a registration wall"""
    # Check for Zephr registration form
    if soup.find("div", class_="zephr-registration-form"):
        return True
    
    if soup.find("div", id=lambda x: x and x.startswith("zephr-form-")):
        return True
    
    # Check for registration wall text
    regwall = soup.find("div", class_="regwall-main")
    if regwall:
        text = regwall.get_text(" ", strip=True).lower()
        if any(phrase in text for phrase in [
            "register below to read",
            "register to gain",
            "subscribe to unlock",
        ]):
            return True
    
    return False


def extract_article(url):
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    # Check for registration wall
    if is_registration_walled(soup):
        print(f"Skipping registration-walled article: {url}")
        return None

    crawl_timestamp = datetime.now(timezone.utc).isoformat()

    # Generate ID from URL slug hash
    article_id = get_slug_hash(url)

    # Headline - h1 with class b-headline
    headline = None
    headline_tag = soup.find("h1", class_="b-headline")
    if headline_tag:
        headline = normalize_text(headline_tag.get_text(strip=True))
        
    # Published date - time tag with class PublicationTime__pub-time
    published = None
    published_date = None
    published_time = None
    published_datetime_raw = None

    time_tag = soup.find("time", class_="PublicationTime__pub-time")
    if time_tag:
        # Get the datetime attribute (raw format)
        published_datetime_raw = time_tag.get("datetime")
        
        # Extract just the date and time parts from spans
        date_span = time_tag.find("span", class_="PublicationTime__date")
        time_span = time_tag.find("span", class_="PublicationTime__time")
        
        if date_span:
            published_date = date_span.get_text(strip=True)
        
        if time_span:
            published_time = time_span.get_text(strip=True)
        
        # Convert to ISO-8601 UTC format
        if date_span and time_span:
            date_text = date_span.get_text(strip=True)
            time_text = time_span.get_text(strip=True)
            full_datetime = f"{date_text}{time_text}"
            published = normalize_published(full_datetime)

    # Article body
    body_text = normalize_text(extract_body_from_html(soup))
    word_count = len(body_text.split())

    if word_count < 120:
        print(f"Skipping short article ({word_count} words): {url}")
        return None

    return {
        "id": article_id,
        "url": url,
        "headline": headline,
        "published_datetime_raw": published_datetime_raw,
        "body": body_text,
        "word_count": word_count,
        "crawl_timestamp": crawl_timestamp,
    }

# --------------------
# Main execution
# --------------------

if __name__ == "__main__":
    # Daily Beast sitemaps - update these URLs as needed
    START_SITEMAPS = [
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/latest/",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-10/?outputType=xml",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-09/?outputType=xml",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-08/?outputType=xml",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-07/?outputType=xml",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-06/?outputType=xml",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-05/?outputType=xml",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-04/?outputType=xml",
        "https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/2026-04-03/?outputType=xml",
    ]

    OUTPUT_FILE = "dailybeast_200_articles.json"

    articles = []
    attempts = 0
    seen_urls = set()

    print(f"Target: {MAX_URLS} valid articles\n")

    for sitemap_url in START_SITEMAPS:
        if len(articles) >= MAX_URLS:
            print(f"\n✓ Target reached: {len(articles)}/{MAX_URLS} articles collected")
            break

        print(f"\nProcessing sitemap: {sitemap_url}")
        
        try:
            entries = crawl_sitemap(sitemap_url)
            entries.sort(key=lambda x: x[1], reverse=True)
            urls = [u for u, _ in entries]
            
            # Filter out already seen URLs
            new_urls = [u for u in urls if u not in seen_urls]
            
            print(f"Found {len(new_urls)} new URLs in this sitemap")
            
            for url in new_urls:
                if len(articles) >= MAX_URLS:
                    break
                
                seen_urls.add(url)
                attempts += 1
                print(f"[{attempts}] {url}")
                
                try:
                    article = extract_article(url)
                    if article:
                        articles.append(article)
                        print(f"✓ Collected {len(articles)}/{MAX_URLS}")
                except Exception as e:
                    print(f"Failed: {e}")
                
                time.sleep(CRAWL_DELAY)
        
        except Exception as e:
            print(f"Error processing sitemap {sitemap_url}: {e}")
            continue

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)

    print(f"\n{'='*60}")
    print(f"SUMMARY")
    print(f"{'='*60}")
    print(f"Total URLs attempted: {attempts}")
    print(f"Valid articles collected: {len(articles)}")
    print(f"Success rate: {len(articles)/attempts*100:.1f}%" if attempts > 0 else "N/A")
    print(f"Output saved to: {OUTPUT_FILE}")
    print(f"{'='*60}\n")

Target: 200 valid articles


Processing sitemap: https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/latest/
Fetching sitemap: https://www.thedailybeast.com/arc/outboundfeeds/sitemap-news/latest/
Found 64 new URLs in this sitemap
[1] https://www.thedailybeast.com/why-melania-trumps-epstein-congress-bomb-is-only-the-beginning/
✓ Collected 1/200
[2] https://www.thedailybeast.com/heres-why-whoops-wellness-wearable-is-a-favorite-among-pga-athletes/
Skipping short article (0 words): https://www.thedailybeast.com/heres-why-whoops-wellness-wearable-is-a-favorite-among-pga-athletes/
[3] https://www.thedailybeast.com/trump-unveils-massive-arch-that-will-dwarf-lincoln-memorial/
✓ Collected 2/200
[4] https://www.thedailybeast.com/obsessed/shrinking-just-aired-the-best-tv-finale-of-the-year/
✓ Collected 3/200
[5] https://www.thedailybeast.com/firebomb-attack-targets-home-of-openai-chief-sam-altman/
Skipping short article (0 words): https://www.thedailybeast.com/firebomb-attack-targets-hom